In [1]:
import requests
import pandas as pd
from tqdm.notebook import tqdm
import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
# --- CONFIGURATION ---
INPUT_FILE = "./geocoded_irish_addresses.csv"
OUTPUT_FILE = "./geocoded_results_final.csv"
MAPBOX_TOKEN = os.getenv('MAPBOX_TOKEN')
CHECKPOINT_INTERVAL = 100  # Save to disk every 100 new matches

In [3]:
# 1. LOAD DATA
# Check if we already have a progress file; if so, resume from there
if os.path.exists(OUTPUT_FILE):
    print(f"Resuming from existing output file: {OUTPUT_FILE}")
    df = pd.read_csv(OUTPUT_FILE)
else:
    print(f"Starting fresh from: {INPUT_FILE}")
    df = pd.read_csv(INPUT_FILE)

Resuming from existing output file: ./geocoded_results_final.csv


In [4]:
# 2. IDENTIFY REMAINING GAPS
# A 'gap' is any row where lat, lon, or matched_address is missing
target_cols = ['lat', 'lon', 'matched_address']
for col in target_cols:
    if col not in df.columns:
        df[col] = None

mask_incomplete = df[target_cols].isnull().any(axis=1)
gaps_df = df[mask_incomplete].copy()

# Deduplicate addresses to save API credits
unique_gaps = gaps_df[['Address', 'County']].drop_duplicates()
total_to_do = len(unique_gaps)

print(f"Total rows in dataset: {len(df)}")
print(f"Unique addresses left to geocode: {total_to_do}")

Total rows in dataset: 54000
Unique addresses left to geocode: 14817


In [ ]:
# 3. THE FAIL-SAFE LOOP
results_cache = {}
new_match_count = 0

try:
    pbar = tqdm(unique_gaps.itertuples(), total=total_to_do, desc="Geocoding")
    
    for row in pbar:
        query = f"{row.Address}, {row.County}, Ireland"
        
        try:
            url = "https://api.mapbox.com/search/geocode/v6/forward"
            resp = requests.get(url, params={
                "q": query, 
                "access_token": MAPBOX_TOKEN, 
                "limit": 1
            })
            data = resp.json()
            
            if data.get("features"):
                feat = data["features"][0]
                props = feat.get("properties", {})
                coords = props.get("coordinates", {})
                
                lat, lon = coords.get("latitude"), coords.get("longitude")
                matched = props.get("full_address")
                
                # Update the main dataframe for all rows matching this address
                addr_mask = (df['Address'] == row.Address) & (df['County'] == row.County)
                df.loc[addr_mask, 'lat'] = lat
                df.loc[addr_mask, 'lon'] = lon
                df.loc[addr_mask, 'matched_address'] = matched
                
                new_match_count += 1
                
            # Periodic Checkpoint Saving
            if new_match_count > 0 and new_match_count % CHECKPOINT_INTERVAL == 0:
                df.to_csv(OUTPUT_FILE, index=False)
            
            pbar.set_postfix(success_rate=f"{(new_match_count/(pbar.n+1))*100:.1f}%")

        except Exception as e:
            tqdm.write(f"Error on {row.Address}: {e}")
            continue

except KeyboardInterrupt:
    print("\n[!] Loop interrupted by user. Saving current progress...")

finally:
    # 4. FINAL SAVE
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Process complete. Data saved to {OUTPUT_FILE}")
    
    # Quick Cardinality View of what's left
    remaining = df['lat'].isnull().sum()
    print(f"Remaining null lats: {remaining} out of {len(df)}")

Geocoding:   0%|          | 0/14817 [00:00<?, ?it/s]